# Lab 08a (Career edition) — NYC Career Event Finder
## Ask about someone's career, then find the fairs, conferences and events that fit

This notebook turns the Food Analyzer lab into a **career-event matching app**. The app:

* asks what the user is looking for — **networking, job hunting, internships, learning new skills,
  exploring a career, mentorship, or a career change**;
* asks for their **job title or major**, and their **year in college** if they are a student;
* ranks career fairs, conferences, networking nights, workshops and info sessions to fit, with a
  **% match and plain-language reasons** on every card;
* uses your **LM Studio** chat model to write a short personalized plan (top picks, prep tips, an
  elevator pitch), and links to **live searches for real upcoming events** (Google, Eventbrite,
  Meetup, LinkedIn, and Handshake for students).

The layout follows the reference design: purple hero banner, a search bar with borough and price
filters, category chips, event cards with Save buttons, and a **"Your career agent"** sidebar with a
Saved events list.

> **About the events:** the 24 events in Cell 3 are **sample events** (like the "12 sample events" in the
> design), so the app works offline and in class. Each card has a **"Find real events like this"** link,
> and the agent's plan ends with search links for real listings. Add or edit events in `EVENTS` in Cell 3.

### What changed from the food lab

| Food lab | Career edition |
| --- | --- |
| CLIP image model + PyTorch (~600 MB download) | Removed — no image model needed, so setup is much faster |
| Nutrition lookup table | Career-event catalog + job-title/major → field detection |
| CLIP label ranking | Transparent scoring: goals, field, college year, time of day, extra notes |
| LM Studio writes a recipe | LM Studio writes a career plan (built-in fallback if the server is off) |
| Two-column Gradio form | Custom-styled event-discovery UI |

### Kernel

Use the same **Python (Nutritionist 3.11)** kernel you already created for the food lab — it already has
Gradio, pandas and requests, so Cell 1 normally installs nothing. If you are starting fresh, run these
in Terminal once:

```bash
conda create --name nutritionist311 python=3.11 pip ipykernel -y
conda run -n nutritionist311 python -m ipykernel install --user --name nutritionist311 --display-name "Python (Nutritionist 3.11)"
```

### Run it

1. **Kernel → Change Kernel → Python (Nutritionist 3.11)**, then run **Cell 1** (must print **SETUP PASSED**).
2. Optional but recommended: in LM Studio, load a **chat/instruct** model and start the local server on
   port **1234**. Run **Cell 2**. If LM Studio is off, the app still works with built-in advice.
3. Run **Cells 3–6** in order. Cell 4 must print **MATCHER PASSED**. Cell 6 opens the app in your browser
   (usually `http://127.0.0.1:7860`).
4. In the app: fill in the **career agent** on the right, click **✦ Find my matches**, then browse,
   filter and **☆ Save** events.

In [ ]:
# === Cell 1: Setup check for this notebook's kernel ===
# Only lightweight packages are needed now (no PyTorch / CLIP).
import os
import sys
import subprocess
import importlib
import importlib.metadata as metadata
from packaging.requirements import Requirement

SETUP_READY = False
print('Notebook Python:', sys.version.split()[0])
print('Executable:', sys.executable)

if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(
        'Select the kernel "Python (Nutritionist 3.11)" (any Python 3.10-3.12 also works).\n'
        f'This kernel is Python {sys.version.split()[0]}; the pinned packages do not ship '
        'ready-made wheels for it. Follow the Terminal commands above, then change the kernel.'
    )

if globals().get('_SETUP_RESTART_REQUIRED', False):
    raise RuntimeError('Packages changed while libraries were loaded. '
                       'Choose Kernel > Restart Kernel, then Run All.')

REQUIREMENTS = ['numpy==1.26.4', 'pandas==2.2.3', 'gradio==5.49.1', 'gradio-client==1.13.3',
                'pydantic==2.11.10', 'requests==2.32.5']

def requirements_to_install():
    missing = []
    for text in REQUIREMENTS:
        requirement = Requirement(text)
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            missing.append(text)
            continue
        if not requirement.specifier.contains(installed):
            missing.append(text)
    return missing

needed = requirements_to_install()
if needed:
    print('\nInstalling:', ', '.join(needed))
    loaded_before_install = any(name in sys.modules for name in ('numpy', 'pandas', 'gradio', 'pydantic', 'requests'))
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--only-binary=:all:',
                        '--disable-pip-version-check', *REQUIREMENTS], check=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError('Installation failed; read the pip error above. Confirm the kernel and '
                           'internet access, then rerun Cell 1.') from error
    importlib.invalidate_caches()
    if loaded_before_install:
        _SETUP_RESTART_REQUIRED = True
        raise RuntimeError('Installation finished. One-time step: Kernel > Restart Kernel, then Run All.')
else:
    print('\nCompatible package versions already installed; skipping installation.')

try:
    import numpy as np
    import pandas as pd
    import gradio as gr
    import requests
except Exception as error:
    raise RuntimeError(f'Package import check failed. Restart the kernel and Run All. Error: {error}') from error

remaining = requirements_to_install()
if remaining:
    raise RuntimeError(f'These package versions still do not match: {remaining}')
for package in ('gradio', 'pandas', 'requests'):
    print(f'  {package}: {metadata.version(package)}')
SETUP_READY = True
print('\nSETUP PASSED. Continue to Cell 2.')

## Cell 2 — Connect to your local LM Studio server (optional)
Rerun this cell if you change the server or chat model.

In [ ]:
# === Cell 2: Connect to LM Studio (optional - the app falls back to built-in advice) ===
if not globals().get('SETUP_READY', False):
    raise RuntimeError('Complete Cell 1 successfully before running Cell 2.')

import json
import requests

LMSTUDIO_BASE_URL = 'http://localhost:1234/v1'
LMSTUDIO_API_KEY = os.environ.get('LMSTUDIO_API_KEY', 'lmstudio-key')
MODEL_ID = ''  # Leave blank for one chat model; otherwise paste a printed ID.
LMSTUDIO_READY = False

headers = {'Authorization': f'Bearer {LMSTUDIO_API_KEY}'}
try:
    response = requests.get(f'{LMSTUDIO_BASE_URL.rstrip("/")}/models', headers=headers, timeout=10)
    response.raise_for_status()
    model_ids = [item['id'] for item in response.json().get('data', []) if item.get('id')]
except (requests.RequestException, ValueError) as error:
    model_ids = []
    print(f'WARNING: cannot reach LM Studio at {LMSTUDIO_BASE_URL} ({error}).\n'
          'The app will still work, using built-in advice instead of the AI-written plan.\n'
          'To enable AI advice: open LM Studio, load a chat model, start the server on port 1234, '
          'then rerun THIS cell (and Cell 6).')

if model_ids:
    print('Model IDs reported by LM Studio:')
    for model_id in model_ids:
        print(' -', model_id)
    if not MODEL_ID:
        chat_candidates = [m for m in model_ids if 'embed' not in m.lower()]
        if len(chat_candidates) == 1:
            MODEL_ID = chat_candidates[0]
        else:
            print(f'\nWARNING: found {len(chat_candidates)} possible chat models. Paste one into '
                  'MODEL_ID above and rerun Cell 2. Using built-in advice until then.')
    if MODEL_ID and MODEL_ID not in model_ids:
        raise RuntimeError(f'MODEL_ID {MODEL_ID!r} is not one of the IDs printed above.')
    if MODEL_ID:
        os.environ['OPENAI_BASE_URL'] = LMSTUDIO_BASE_URL.rstrip('/')
        os.environ['OPENAI_API_KEY'] = LMSTUDIO_API_KEY
        os.environ['MODEL'] = MODEL_ID
        LMSTUDIO_READY = True
        print('\nLM Studio is reachable. Selected model:', MODEL_ID)

## Cell 3 — Career-event catalog and profile helpers
Sample NYC events plus the keyword map that turns a job title or major into career fields.

In [ ]:
# === Cell 3: Sample career-event catalog and profile helpers ===
# These are SAMPLE events for the lab (like the "12 sample events" in the design).
# Each card's "Find real events like this" link opens a live search for similar
# real events, so users can move from the demo to actual listings.
from typing import List, Dict, Any
import re
import html
import urllib.parse
import pandas as pd

CATEGORIES = ["Job fair", "Conference", "Networking", "Workshop", "Info session"]
CATEGORY_ICONS = {
    "Job fair": "💼", "Conference": "🎤", "Networking": "🤝",
    "Workshop": "🛠️", "Info session": "🏫",
}
BOROUGHS = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "Online"]

# What the user is looking for -> internal goal tags used by events.
GOAL_OPTIONS = {
    "Networking": "networking",
    "Job hunting": "job_hunting",
    "Internships": "internships",
    "Learning new skills": "learning",
    "Exploring a career / learning more": "exploring",
    "Mentorship": "mentorship",
    "Career change": "career_change",
}
COLLEGE_YEARS = [
    "Not in college", "Freshman", "Sophomore", "Junior", "Senior",
    "Graduate student", "Recent graduate (0-2 yrs)",
]
TIME_OPTIONS = ["Anytime", "Weekday evenings", "Weekends", "Weekday daytime"]

# Fields: tech, healthcare, finance, marketing, design, engineering, education,
# law, nonprofit, media, science, any. Audience: students, early, experienced, all.
EVENTS: List[Dict[str, Any]] = [
    {"id": 1, "title": "Tech & Startup Career Mixer", "category": "Networking",
     "borough": "Manhattan", "venue": "Midtown community space", "day": "Tue", "time": "6:00 PM",
     "slot": "evening", "price": 0, "fields": ["tech"], "goals": ["networking", "job_hunting"],
     "audience": ["early", "experienced", "students"],
     "desc": "Meet recruiters, startup founders and peers exploring the NYC tech scene."},
    {"id": 2, "title": "Healthcare Careers Open House", "category": "Info session",
     "borough": "Queens", "venue": "Queens training center", "day": "Thu", "time": "1:00 PM",
     "slot": "daytime", "price": 0, "fields": ["healthcare"], "goals": ["exploring", "job_hunting", "career_change"],
     "audience": ["all"],
     "desc": "Learn about entry-level roles, training paths and hiring resources."},
    {"id": 3, "title": "NYC Campus-Wide Internship Fair", "category": "Job fair",
     "borough": "Manhattan", "venue": "University student center", "day": "Wed", "time": "11:00 AM",
     "slot": "daytime", "price": 0, "fields": ["any"], "goals": ["internships", "job_hunting"],
     "audience": ["students"],
     "desc": "100+ employers hiring summer interns across business, tech, media and nonprofits."},
    {"id": 4, "title": "Data & AI Summit NYC", "category": "Conference",
     "borough": "Manhattan", "venue": "Hudson Yards convention hall", "day": "Fri", "time": "9:00 AM",
     "slot": "daytime", "price": 249, "fields": ["tech", "science"], "goals": ["learning", "networking"],
     "audience": ["early", "experienced"],
     "desc": "Talks on machine learning, analytics and AI in production, plus a hiring expo."},
    {"id": 5, "title": "Women in Finance Breakfast", "category": "Networking",
     "borough": "Manhattan", "venue": "FiDi conference center", "day": "Thu", "time": "8:00 AM",
     "slot": "daytime", "price": 25, "fields": ["finance"], "goals": ["networking", "mentorship"],
     "audience": ["early", "experienced", "students"],
     "desc": "Panel and speed-mentoring with analysts, PMs and bankers from NYC firms."},
    {"id": 6, "title": "Brooklyn Creative Portfolio Review", "category": "Workshop",
     "borough": "Brooklyn", "venue": "DUMBO design studio", "day": "Sat", "time": "12:00 PM",
     "slot": "weekend", "price": 0, "fields": ["design", "media", "marketing"], "goals": ["mentorship", "job_hunting", "learning"],
     "audience": ["students", "early"],
     "desc": "Bring your portfolio for 1:1 feedback from art directors and UX leads."},
    {"id": 7, "title": "Resume & LinkedIn Glow-Up Clinic", "category": "Workshop",
     "borough": "Online", "venue": "Live virtual session", "day": "Mon", "time": "7:00 PM",
     "slot": "evening", "price": 0, "fields": ["any"], "goals": ["job_hunting", "career_change", "internships"],
     "audience": ["all"],
     "desc": "Recruiters rewrite real resumes live and share what gets interviews."},
    {"id": 8, "title": "Marketing & Brand Leaders Conference", "category": "Conference",
     "borough": "Manhattan", "venue": "Chelsea event loft", "day": "Wed", "time": "10:00 AM",
     "slot": "daytime", "price": 149, "fields": ["marketing", "media"], "goals": ["learning", "networking"],
     "audience": ["early", "experienced"],
     "desc": "Growth, content and brand strategy sessions with agency and in-house leaders."},
    {"id": 9, "title": "Bronx Community Job Fair", "category": "Job fair",
     "borough": "Bronx", "venue": "Bronx community college gym", "day": "Sat", "time": "10:00 AM",
     "slot": "weekend", "price": 0, "fields": ["any", "healthcare", "education"], "goals": ["job_hunting", "career_change"],
     "audience": ["all"],
     "desc": "Employers hiring on the spot in healthcare, retail, transit, city agencies and more."},
    {"id": 10, "title": "Software Engineering Interview Prep Night", "category": "Workshop",
     "borough": "Brooklyn", "venue": "Downtown Brooklyn coworking space", "day": "Tue", "time": "6:30 PM",
     "slot": "evening", "price": 0, "fields": ["tech"], "goals": ["job_hunting", "internships", "learning"],
     "audience": ["students", "early"],
     "desc": "Mock technical interviews, LeetCode-style practice and tips from engineers."},
    {"id": 11, "title": "Nursing & Allied Health Hiring Day", "category": "Job fair",
     "borough": "Brooklyn", "venue": "Hospital education center", "day": "Fri", "time": "10:00 AM",
     "slot": "daytime", "price": 0, "fields": ["healthcare"], "goals": ["job_hunting"],
     "audience": ["students", "early", "experienced"],
     "desc": "Meet nurse recruiters and hiring managers; bring license info and resumes."},
    {"id": 12, "title": "Pre-Law Pathways Panel", "category": "Info session",
     "borough": "Manhattan", "venue": "Law school auditorium", "day": "Thu", "time": "5:30 PM",
     "slot": "evening", "price": 0, "fields": ["law"], "goals": ["exploring", "mentorship"],
     "audience": ["students"],
     "desc": "Admissions officers and young attorneys on LSAT prep, applications and paralegal work."},
    {"id": 13, "title": "Career Changers Meetup: Pivot Into Tech", "category": "Networking",
     "borough": "Queens", "venue": "Long Island City café", "day": "Wed", "time": "7:00 PM",
     "slot": "evening", "price": 0, "fields": ["tech"], "goals": ["career_change", "networking", "exploring"],
     "audience": ["experienced", "early"],
     "desc": "Bootcamp grads and hiring managers share what a mid-career pivot really takes."},
    {"id": 14, "title": "Teach NYC Recruitment Fair", "category": "Job fair",
     "borough": "Manhattan", "venue": "Education department hall", "day": "Sat", "time": "11:00 AM",
     "slot": "weekend", "price": 0, "fields": ["education"], "goals": ["job_hunting", "career_change"],
     "audience": ["students", "early", "experienced"],
     "desc": "Public and charter schools hiring teachers, aides and counselors; certification help."},
    {"id": 15, "title": "Civil & Mechanical Engineering Expo", "category": "Conference",
     "borough": "Queens", "venue": "Queens exhibition center", "day": "Thu", "time": "9:30 AM",
     "slot": "daytime", "price": 40, "fields": ["engineering"], "goals": ["learning", "networking", "job_hunting"],
     "audience": ["students", "early", "experienced"],
     "desc": "Infrastructure, construction and energy firms showcase projects and open roles."},
    {"id": 16, "title": "Nonprofit & Public Service Careers Night", "category": "Networking",
     "borough": "Brooklyn", "venue": "Brooklyn public library branch", "day": "Mon", "time": "6:00 PM",
     "slot": "evening", "price": 0, "fields": ["nonprofit", "education", "law"], "goals": ["networking", "exploring", "job_hunting"],
     "audience": ["all"],
     "desc": "Meet NGOs, foundations and city agencies working on housing, health and education."},
    {"id": 17, "title": "Fintech & Accounting Career Expo", "category": "Job fair",
     "borough": "Manhattan", "venue": "Midtown hotel ballroom", "day": "Tue", "time": "12:00 PM",
     "slot": "daytime", "price": 0, "fields": ["finance", "tech"], "goals": ["job_hunting", "internships"],
     "audience": ["students", "early"],
     "desc": "Banks, fintech startups and Big Four firms recruiting analysts and interns."},
    {"id": 18, "title": "UX Research & Product Design Bootcamp Day", "category": "Workshop",
     "borough": "Online", "venue": "Live virtual workshop", "day": "Sat", "time": "1:00 PM",
     "slot": "weekend", "price": 35, "fields": ["design", "tech"], "goals": ["learning", "career_change"],
     "audience": ["all"],
     "desc": "Hands-on intro to user research, wireframing and building a case study."},
    {"id": 19, "title": "Biotech & Life Sciences Networking", "category": "Networking",
     "borough": "Manhattan", "venue": "East Side life-science hub", "day": "Thu", "time": "6:00 PM",
     "slot": "evening", "price": 15, "fields": ["science", "healthcare"], "goals": ["networking", "exploring", "job_hunting"],
     "audience": ["students", "early", "experienced"],
     "desc": "Researchers, lab managers and biotech recruiters mingle over lightning talks."},
    {"id": 20, "title": "First-Gen Student Mentorship Brunch", "category": "Networking",
     "borough": "Staten Island", "venue": "Staten Island college hall", "day": "Sun", "time": "11:00 AM",
     "slot": "weekend", "price": 0, "fields": ["any"], "goals": ["mentorship", "exploring", "networking"],
     "audience": ["students"],
     "desc": "Pairs first-generation students with professionals across many industries."},
    {"id": 21, "title": "Media, Journalism & Content Careers Panel", "category": "Info session",
     "borough": "Brooklyn", "venue": "Williamsburg media studio", "day": "Wed", "time": "6:30 PM",
     "slot": "evening", "price": 10, "fields": ["media", "marketing"], "goals": ["exploring", "networking"],
     "audience": ["students", "early"],
     "desc": "Editors, producers and creators on breaking into newsrooms and creator teams."},
    {"id": 22, "title": "Cybersecurity Career Day", "category": "Conference",
     "borough": "Online", "venue": "Virtual expo hall", "day": "Fri", "time": "11:00 AM",
     "slot": "daytime", "price": 0, "fields": ["tech"], "goals": ["learning", "job_hunting", "career_change"],
     "audience": ["all"],
     "desc": "Certification roadmaps, SOC analyst demos and a virtual hiring booth."},
    {"id": 23, "title": "Leadership Lab for New Managers", "category": "Workshop",
     "borough": "Manhattan", "venue": "Flatiron training room", "day": "Wed", "time": "6:00 PM",
     "slot": "evening", "price": 79, "fields": ["any"], "goals": ["learning", "mentorship"],
     "audience": ["experienced"],
     "desc": "Feedback, delegation and one-on-ones for people managing a team for the first time."},
    {"id": 24, "title": "Graduate School & Fellowship Fair", "category": "Info session",
     "borough": "Manhattan", "venue": "Upper West Side campus", "day": "Sat", "time": "10:00 AM",
     "slot": "weekend", "price": 0, "fields": ["any", "science", "law", "education"], "goals": ["exploring", "learning"],
     "audience": ["students"],
     "desc": "Meet master's, PhD and fellowship programs; funding and application Q&A."},
]
EVENTS_BY_ID = {event["id"]: event for event in EVENTS}

# Job title / major keywords -> fields. Longer phrases are checked first.
FIELD_KEYWORDS = {
    "tech": ["computer science", "software", "developer", "programmer", "data scien", "data analy",
             "data engineer", "machine learning", "information technology", "information systems",
             "cyber", "web dev", "devops", "cloud", "it support", "comp sci", "cs ", "ai ", "product manager"],
    "healthcare": ["nurs", "pre-med", "premed", "medic", "health", "pharm", "biology", "physician",
                   "physical therap", "dental", "kinesiology", "public health", "emt"],
    "finance": ["financ", "accounting", "accountant", "economics", "banking", "investment", "actuar",
                "business admin", "fintech", "cpa"],
    "marketing": ["marketing", "communications", "advertis", "public relations", " pr ", "social media",
                  "brand", "sales", "seo", "content"],
    "design": ["design", "ux", "ui ", "graphic", "illustrat", "fine art", "architecture", "animation", "fashion"],
    "engineering": ["mechanical", "civil engineer", "electrical", "chemical engineer", "industrial engineer",
                    "aerospace", "construction", "engineering"],
    "education": ["education", "teach", "tutor", "school counsel", "early childhood"],
    "law": ["law", "legal", "paralegal", "political science", "criminal justice", "attorney"],
    "nonprofit": ["nonprofit", "non-profit", "social work", "public policy", "urban planning", "sociology"],
    "media": ["journalism", "film", "media", "writer", "writing", "english", "photograph", "music"],
    "science": ["chemistry", "physics", "biochem", "neuroscience", "biotech", "math", "statistics",
                "research", "environmental"],
}
FIELD_LABELS = {
    "tech": "tech", "healthcare": "healthcare", "finance": "finance & business",
    "marketing": "marketing & communications", "design": "design & creative",
    "engineering": "engineering", "education": "education", "law": "law & policy",
    "nonprofit": "nonprofit & public service", "media": "media & writing",
    "science": "science & research",
}

def detect_fields(text: str) -> List[str]:
    """Map a free-text job title or major (e.g. 'Junior nursing major') to fields."""
    padded = f" {text.lower().strip()} "
    found = []
    for field, keywords in FIELD_KEYWORDS.items():
        if any(keyword in padded for keyword in keywords):
            found.append(field)
    # "software engineer" is tech, not civil/mechanical engineering.
    if "tech" in found and "engineering" in found and not any(
        word in padded for word in ["mechanical", "civil", "electrical", "chemical", "industrial", "aerospace"]
    ):
        found.remove("engineering")
    return found

def audience_for(college_year: str) -> str:
    if college_year in ("Freshman", "Sophomore", "Junior", "Senior", "Graduate student"):
        return "students"
    if college_year == "Recent graduate (0-2 yrs)":
        return "early"
    return "experienced"

def price_label(event: Dict[str, Any]) -> str:
    return "Free" if event["price"] == 0 else f"${event['price']}"

def real_event_search_url(event: Dict[str, Any]) -> str:
    """Live web search for real events similar to this sample card."""
    query = f"{event['title']} {event['borough'] if event['borough'] != 'Online' else 'virtual'} NYC 2026"
    return "https://www.google.com/search?q=" + urllib.parse.quote_plus(query)

def events_df() -> pd.DataFrame:
    return pd.DataFrame(EVENTS)[["id", "title", "category", "borough", "day", "time", "price"]]

print(f"Career-event catalog loaded: {len(EVENTS)} sample NYC events.")
print("Field detection check:", detect_fields("Junior, computer science major"))

## Cell 4 — Matching engine, card design, and self-test
Wait for **MATCHER PASSED** before continuing.

In [ ]:
# === Cell 4: Matching engine, card rendering, and a self-test ===
# The ranking is transparent rule-based scoring, so every match comes with
# plain-language reasons. The LLM (Cell 5) only writes advice on top of it.

STOPWORDS = set("""a an and the for with that this from into about your have want
looking find some more like also just really would could i'm im i me my to of in on
at or be is are was am it its as by events event""".split())
TIME_TO_SLOT = {"Weekday evenings": "evening", "Weekends": "weekend", "Weekday daytime": "daytime"}
SLOT_WORDS = {"evening": "after work", "weekend": "on a weekend", "daytime": "during the day"}

def build_profile(goal_labels: List[str], title_or_major: str, college_year: str,
                  best_time: str = "Anytime", notes: str = "") -> Dict[str, Any]:
    title_or_major = (title_or_major or "").strip()
    notes = (notes or "").strip()
    goals = [GOAL_OPTIONS[label] for label in (goal_labels or []) if label in GOAL_OPTIONS]
    fields = detect_fields(title_or_major) or detect_fields(notes)
    words = {w for w in re.findall(r"[a-z]+", notes.lower()) if len(w) > 3 and w not in STOPWORDS}
    return {
        "goal_labels": list(goal_labels or []), "goals": goals,
        "title_or_major": title_or_major, "college_year": college_year or "Not in college",
        "audience": audience_for(college_year or "Not in college"),
        "fields": fields, "best_time": best_time or "Anytime", "notes": notes, "note_words": words,
    }

def score_event(event: Dict[str, Any], profile: Dict[str, Any]):
    score, reasons = 0.0, []

    shared_goals = [g for g in profile["goals"] if g in event["goals"]]
    if shared_goals:
        score += 3 * len(shared_goals)
        names = [label.lower() for label, tag in GOAL_OPTIONS.items() if tag in shared_goals]
        reasons.append("good for " + ", ".join(names))
    elif profile["goals"]:
        score -= 2

    if profile["fields"]:
        shared_fields = [f for f in profile["fields"] if f in event["fields"]]
        if shared_fields:
            score += 5
            reasons.append("matches your field (" + ", ".join(FIELD_LABELS[f] for f in shared_fields) + ")")
        elif "any" in event["fields"]:
            score += 1.5
            reasons.append("open to every major and industry")
        else:
            score -= 3
    elif "any" in event["fields"]:
        score += 1

    audience = profile["audience"]
    if audience in event["audience"] or "all" in event["audience"]:
        score += 2
        if audience == "students" and "students" in event["audience"]:
            reasons.append("welcomes students")
    else:
        score -= 4  # e.g. a manager workshop for a freshman

    year = profile["college_year"]
    if year in ("Freshman", "Sophomore") and set(event["goals"]) & {"exploring", "mentorship", "internships"}:
        score += 1.5
        reasons.append(f"a smart early step for a {year.lower()}")
    elif year in ("Junior", "Senior", "Graduate student") and set(event["goals"]) & {"internships", "job_hunting"}:
        score += 1.5
        reasons.append(f"well timed for a {year.lower()} recruiting season")
    elif year == "Recent graduate (0-2 yrs)" and "job_hunting" in event["goals"]:
        score += 1.5
        reasons.append("hiring for early-career roles")

    wanted_slot = TIME_TO_SLOT.get(profile["best_time"])
    if wanted_slot:
        if event["slot"] == wanted_slot:
            score += 1.5
            reasons.append("happens " + SLOT_WORDS[wanted_slot])
        else:
            score -= 1

    text = f"{event['title']} {event['desc']} {event['category']}".lower()
    hits = [w for w in profile["note_words"] if w in text][:3]
    if hits:
        score += len(hits)
        reasons.append("mentions " + ", ".join(hits))

    best_possible = 3 * max(len(profile["goals"]), 1) + 5 + 2 + 1.5 + 1.5 + 2
    percent = int(max(0, min(99, round(100 * score / best_possible))))
    return score, percent, reasons

def rank_events(profile: Dict[str, Any], events: List[Dict[str, Any]] | None = None, top_k: int = 8):
    ranked = []
    for event in events if events is not None else EVENTS:
        score, percent, reasons = score_event(event, profile)
        if score > 0:
            ranked.append({"id": event["id"], "score": score, "percent": percent, "reasons": reasons})
    ranked.sort(key=lambda item: (-item["score"], item["id"]))
    return ranked[:top_k]

def filter_events(search: str = "", borough: str = "All boroughs", price: str = "Any price",
                  category: str = "All events") -> List[Dict[str, Any]]:
    terms = [t for t in re.findall(r"[a-z0-9]+", (search or "").lower()) if t not in STOPWORDS]
    results = []
    for event in EVENTS:
        if borough and borough != "All boroughs" and event["borough"] != borough:
            continue
        if price == "Free" and event["price"] != 0:
            continue
        if price == "Paid" and event["price"] == 0:
            continue
        if category and category not in ("All events", "All") and event["category"] != category:
            continue
        haystack = " ".join([event["title"], event["desc"], event["category"], event["borough"],
                             event["venue"], " ".join(FIELD_LABELS.get(f, f) for f in event["fields"])]).lower()
        if terms and not all(term in haystack for term in terms):
            continue
        results.append(event)
    return results

# ---------- HTML rendering (styled to match the reference design) ----------
def _e(text) -> str:
    return html.escape(str(text))

import math
BURST_SVG = "".join(
    f'<line x1="{40 + 9 * math.cos(i * math.pi / 8):.1f}" y1="{40 + 9 * math.sin(i * math.pi / 8):.1f}" '
    f'x2="{40 + 36 * math.cos(i * math.pi / 8):.1f}" y2="{40 + 36 * math.sin(i * math.pi / 8):.1f}" />'
    for i in range(16)
)

def render_hero() -> str:
    return f"""
<div class="ce-hero">
  <div>
    <h1>More opportunities.<br>More of your kind of people.</h1>
    <p>Discover career fairs, conferences, networking nights and workshops across
    New York City &mdash; tailored to your major, your role and your goals.</p>
  </div>
  <svg class="ce-burst" viewBox="0 0 80 80" aria-hidden="true"><g>{BURST_SVG}</g></svg>
</div>"""

def render_card(event: Dict[str, Any], match: Dict[str, Any] | None, saved: bool) -> str:
    price = price_label(event)
    price_class = "free" if event["price"] == 0 else "paid"
    why = ""
    if match:
        reasons = "; ".join(match["reasons"][:3]) or "fits your profile"
        why = (f'<div class="ce-why"><span class="ce-pct">{match["percent"]}% match</span>'
               f'<span>{_e(reasons[0].upper() + reasons[1:])}</span></div>')
    star, label = ("★", "Saved") if saved else ("☆", "Save")
    onclick = (
        "(function(){var r=document.querySelector('#ce-save-box');"
        "var t=r&&r.querySelector('textarea,input');if(!t)return;"
        f"t.value='{event['id']}:'+Date.now();"
        "t.dispatchEvent(new Event('input',{bubbles:true}));})()"
    )
    return f"""
<div class="ce-card{' ce-matched' if match else ''}">
  <div class="ce-card-top">
    <span class="ce-tag">{CATEGORY_ICONS[event['category']]} {_e(event['category'])}</span>
    <span class="ce-price {price_class}">{price}</span>
  </div>
  <h3>{_e(event['title'])}</h3>
  <div class="ce-meta">📍 {_e(event['borough'])} · {_e(event['venue'])}</div>
  <div class="ce-meta">🗓️ Sample · {_e(event['day'])} {_e(event['time'])}</div>
  <p class="ce-desc">{_e(event['desc'])}</p>
  {why}
  <div class="ce-card-foot">
    <a class="ce-discover" href="{_e(real_event_search_url(event))}" target="_blank" rel="noopener">✦ Find real events like this</a>
    <button class="ce-save{' on' if saved else ''}" onclick="{_e(onclick)}">{star} {label}</button>
  </div>
</div>"""

def render_grid(events: List[Dict[str, Any]], ranked: List[Dict[str, Any]] | None, saved_ids: List[int]) -> str:
    match_by_id = {m["id"]: m for m in (ranked or [])}
    if ranked:
        visible = [EVENTS_BY_ID[m["id"]] for m in ranked if EVENTS_BY_ID[m["id"]] in events]
        heading = "Your top matches"
        note = f"{len(visible)} matched · ranked for you"
    else:
        visible = events
        heading = "Explore career opportunities"
        note = f"{len(visible)} sample events"
    if not visible:
        body = ('<div class="ce-empty">No events match these filters. Try another borough, '
                'price, or category &mdash; or clear the search box.</div>')
    else:
        body = '<div class="ce-grid">' + "".join(
            render_card(e, match_by_id.get(e["id"]), e["id"] in saved_ids) for e in visible) + "</div>"
    return f'<div class="ce-section-head"><h2>{heading}</h2><span>{note}</span></div>{body}'

def render_saved(saved_ids: List[int]) -> str:
    head = f'<div class="ce-section-head ce-saved-head"><h2>Saved events</h2><span>{len(saved_ids)} saved</span></div>'
    if not saved_ids:
        return head + '<div class="ce-saved-empty">Tap ☆ Save on any card to keep it here.</div>'
    items = "".join(
        f'<li><b>{_e(EVENTS_BY_ID[i]["title"])}</b><br><span>{_e(EVENTS_BY_ID[i]["borough"])} · '
        f'{_e(EVENTS_BY_ID[i]["day"])} {_e(EVENTS_BY_ID[i]["time"])} · {price_label(EVENTS_BY_ID[i])}</span></li>'
        for i in saved_ids if i in EVENTS_BY_ID)
    return head + f'<ul class="ce-saved-list">{items}</ul>'

APP_CSS = """
.gradio-container { background:#f4f5fb !important; max-width:1280px !important; margin:0 auto !important;
  font-family: Inter, 'Helvetica Neue', Arial, sans-serif !important; }
footer { display:none !important; }
.ce-hero { display:flex; justify-content:space-between; align-items:center; gap:24px;
  background:linear-gradient(100deg,#2b2766 0%,#4a43a3 55%,#6d5fd6 100%); color:#fff;
  border-radius:0 0 26px 26px; padding:30px 36px 34px; margin-bottom:6px; }
.ce-hero h1 { font-size:42px; line-height:1.12; font-weight:800; letter-spacing:-1.2px; margin:0 0 14px; color:#fff; }
.ce-hero p { font-size:16px; line-height:1.55; color:#e4e2fb; max-width:620px; margin:0; }
.ce-burst { width:72px; height:72px; flex:none; margin-right:10px; }
.ce-burst line { stroke:#fff; stroke-width:2.4; stroke-linecap:round; }
.ce-panel { background:#fff !important; border:1px solid #e6e8f0 !important; border-radius:18px !important;
  padding:22px !important; box-shadow:0 1px 2px rgba(20,20,60,.04); }
.ce-filters input, .ce-filters textarea, .ce-filters .wrap-inner { min-height:44px; font-size:15px !important; }
#ce-chips .wrap { gap:10px !important; }
#ce-chips label { border:1px solid #e3e5ee !important; border-radius:999px !important; background:#fff !important;
  padding:7px 15px !important; font-size:14px !important; color:#3a3f55 !important; box-shadow:none !important; cursor:pointer; }
#ce-chips label.selected, #ce-chips label:has(input:checked) { background:#ece9ff !important;
  border-color:#b9b0f5 !important; color:#4338ca !important; font-weight:600; }
#ce-chips input[type=radio] { display:none !important; }
.ce-section-head { display:flex; justify-content:space-between; align-items:baseline; margin:6px 2px 12px; }
.ce-section-head h2 { font-size:19px; font-weight:700; color:#1d2033; margin:0; }
.ce-section-head span { font-size:13px; color:#6b7085; }
.ce-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(300px,1fr)); gap:16px; }
.ce-card { background:#fff; border:1px solid #e3e6ef; border-radius:16px; padding:18px 20px 16px;
  display:flex; flex-direction:column; transition:box-shadow .15s, border-color .15s; }
.ce-card:hover { border-color:#c9c2f7; box-shadow:0 4px 18px rgba(80,70,200,.08); }
.ce-card.ce-matched { border-color:#c9c2f7; }
.ce-card-top { display:flex; justify-content:space-between; align-items:center; margin-bottom:12px; }
.ce-tag { background:#eef0fb; color:#454a9e; font-size:12.5px; font-weight:600; padding:4px 9px; border-radius:7px; }
.ce-price { font-size:13px; font-weight:700; }
.ce-price.free { color:#2f9e5b; } .ce-price.paid { color:#2f9e5b; }
.ce-card h3 { font-size:18px; font-weight:700; color:#1b1d2a; margin:0 0 10px; line-height:1.3; }
.ce-meta { font-size:13px; color:#5b6070; margin-bottom:7px; }
.ce-desc { font-size:13.5px; color:#555b6e; line-height:1.5; margin:4px 0 12px; flex:1; }
.ce-why { background:#f6f4ff; border-radius:10px; padding:8px 10px; font-size:12.5px; color:#3f3a8a;
  margin-bottom:12px; display:flex; gap:8px; align-items:flex-start; line-height:1.4; }
.ce-pct { background:#4f46e5; color:#fff; font-weight:700; border-radius:6px; padding:1px 7px; white-space:nowrap; }
.ce-card-foot { display:flex; justify-content:space-between; align-items:center; margin-top:auto; }
.ce-discover { color:#4f46e5 !important; font-size:13px; font-weight:600; text-decoration:none !important; }
.ce-discover:hover { text-decoration:underline !important; }
.ce-save { background:#f1f2f6; border:1px solid #e3e5ee; border-radius:9px; padding:7px 14px;
  font-size:14.5px; color:#2b2f45; cursor:pointer; }
.ce-save.on { background:#ece9ff; border-color:#b9b0f5; color:#4338ca; font-weight:600; }
.ce-empty, .ce-saved-empty { color:#6b7085; font-size:14px; padding:18px; background:#fafbfe;
  border:1px dashed #dfe2ec; border-radius:12px; }
.ce-agent-head { display:flex; gap:12px; align-items:center; margin-bottom:10px; }
.ce-agent-icon { width:40px; height:40px; border-radius:11px; background:#ece9ff; color:#4f46e5;
  display:flex; align-items:center; justify-content:center; font-size:19px; }
.ce-agent-head b { font-size:18px; color:#1b1d2a; display:block; }
.ce-agent-head small { color:#6b7085; font-size:12.5px; }
.ce-agent-intro { font-size:13.5px; color:#5b6070; line-height:1.5; margin:0 0 6px; }
.ce-tip { background:#f4f3fd; border-radius:12px; padding:12px 14px; font-size:13.5px; color:#3a3f55; line-height:1.5; }
#ce-find-btn { background:#4f46e5 !important; color:#fff !important; border:none !important; border-radius:12px !important;
  font-size:16px !important; font-weight:700 !important; min-height:46px; }
#ce-find-btn:hover { background:#4338ca !important; }
#ce-advice { background:#fafaff; border:1px solid #ebe8ff; border-radius:12px; padding:4px 14px; }
#ce-save-box { position:absolute !important; left:-9999px !important; width:1px !important; height:1px !important; overflow:hidden !important; }
.ce-saved-head { margin-top:16px; }
.ce-saved-list { list-style:none; padding:0; margin:0; }
.ce-saved-list li { border:1px solid #e6e8f0; border-radius:10px; padding:9px 12px; margin-bottom:8px; font-size:13.5px; color:#1d2033; }
.ce-saved-list li span { color:#6b7085; font-size:12.5px; }
"""

def toggle_saved(saved_ids: List[int], event_id: int) -> List[int]:
    saved = list(saved_ids or [])
    if event_id in saved:
        saved.remove(event_id)
    elif event_id in EVENTS_BY_ID:
        saved.append(event_id)
    return saved

# ---- Self-test: three different users should get sensibly different results ----
_tests = {
    "Junior CS student, internships": build_profile(["Internships", "Job hunting"], "Computer science major", "Junior"),
    "Nurse, job hunting on weekends": build_profile(["Job hunting"], "Registered nurse", "Not in college", "Weekends"),
    "Freshman undecided, exploring": build_profile(["Exploring a career / learning more", "Mentorship"], "Undecided", "Freshman"),
}
for name, prof in _tests.items():
    top = rank_events(prof, top_k=3)
    assert top, f"No matches for test profile: {name}"
    print(f"{name}:")
    for item in top:
        print(f"   {item['percent']:>2}%  {EVENTS_BY_ID[item['id']]['title']}  ({'; '.join(item['reasons'])})")
assert "Leadership Lab for New Managers" not in [EVENTS_BY_ID[m["id"]]["title"] for m in rank_events(_tests["Freshman undecided, exploring"])]
assert len(filter_events(borough="Queens")) > 0 and all(e["price"] == 0 for e in filter_events(price="Free"))
assert "ce-card" in render_grid(EVENTS[:2], None, [])
MATCHER_READY = True
print("\nMATCHER PASSED: ranking, filters and card rendering work.")

## Cell 5 — Career-agent advice
Uses LM Studio when connected; otherwise writes a built-in plan.

In [ ]:
# === Cell 5: Career-agent advice (LM Studio) with a built-in fallback ===
if not globals().get("MATCHER_READY", False):
    raise RuntimeError("Run Cell 4 successfully before Cell 5.")

def lmstudio_chat(system_prompt: str, user_prompt: str) -> str:
    """OpenAI-compatible call to LM Studio (system text folded into one user turn
    for broad local-model compatibility, as in the original lab)."""
    url = f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/chat/completions"
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"}
    payload = {
        "model": os.environ["MODEL"], "temperature": 0.4, "max_tokens": 800,
        "messages": [{"role": "user", "content":
            f"### System instruction (follow strictly):\n{system_prompt}\n\n### User request:\n{user_prompt}"}],
    }
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=(10, 180))
        resp.raise_for_status()
        content = resp.json()["choices"][0]["message"]["content"]
        if not isinstance(content, str) or not content.strip():
            raise ValueError("The chat model returned an empty response.")
        # Some reasoning models wrap hidden thinking in <think> tags; drop it.
        return re.sub(r"<think>.*?</think>", "", content, flags=re.S).strip()
    except (requests.RequestException, ValueError, KeyError, IndexError, TypeError) as error:
        raise RuntimeError(f"LM Studio request failed: {error}") from error

def real_search_links(profile: Dict[str, Any]) -> str:
    """Live searches for REAL upcoming events that fit the profile."""
    cleaned = re.sub(r"\b(freshman|sophomore|junior|senior|major|majoring|student|in|a|an|i'm|im|i am|year)\b|[,.]",
                     " ", profile["title_or_major"].lower())
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    subject = cleaned or (FIELD_LABELS[profile["fields"][0]] if profile["fields"] else "career")
    goal_word = {"job_hunting": "career fair", "internships": "internship fair", "networking": "networking event",
                 "learning": "workshop", "exploring": "career panel", "mentorship": "mentorship event",
                 "career_change": "career change workshop"}
    kind = goal_word.get(profile["goals"][0], "career event") if profile["goals"] else "career event"
    query = f"{subject} {kind}"
    q = urllib.parse.quote_plus
    links = [
        ("Google", "https://www.google.com/search?q=" + q(f"{query} New York City 2026")),
        ("Eventbrite", "https://www.eventbrite.com/d/ny--new-york/" + q(re.sub(r"[^a-z0-9]+", "-", query.lower()).strip("-")) + "/"),
        ("Meetup", "https://www.meetup.com/find/?source=EVENTS&location=us--ny--New%20York&keywords=" + q(query)),
        ("LinkedIn Events", "https://www.linkedin.com/search/results/events/?keywords=" + q(f"{query} New York")),
    ]
    if profile["audience"] == "students":
        links.append(("Handshake (via your school login)", "https://app.joinhandshake.com/stu/events"))
    return " · ".join(f"[{name}]({url})" for name, url in links)

def describe_matches(ranked: List[Dict[str, Any]], limit: int = 5) -> str:
    lines = []
    for m in ranked[:limit]:
        e = EVENTS_BY_ID[m["id"]]
        lines.append(f"- {e['title']} ({e['category']}, {e['borough']}, {e['day']} {e['time']}, "
                     f"{price_label(e)}) - {m['percent']}% match: {'; '.join(m['reasons'])}. {e['desc']}")
    return "\n".join(lines)

def fallback_advice(profile: Dict[str, Any], ranked: List[Dict[str, Any]]) -> str:
    who = profile["title_or_major"] or "you"
    year = "" if profile["college_year"] == "Not in college" else f" ({profile['college_year'].lower()})"
    goals = ", ".join(g.lower() for g in profile["goal_labels"]) or "exploring options"
    top = [EVENTS_BY_ID[m["id"]] for m in ranked[:3]]
    prep = {"Job fair": "bring printed resumes and a 30-second intro",
            "Conference": "pick two sessions in advance and one speaker to follow up with",
            "Networking": "set a goal of three real conversations and connect on LinkedIn the same day",
            "Workshop": "bring your laptop and a current draft to work on",
            "Info session": "prepare two questions you can't answer from the website"}
    picks = "\n".join(f"{i}. **{e['title']}** - {prep[e['category']]}." for i, e in enumerate(top, 1))
    return (f"**Your plan, {who}{year}** - focused on {goals}.\n\n{picks}\n\n"
            f"**Find real upcoming events:** {real_search_links(profile)}")

def build_career_advice(profile: Dict[str, Any], ranked: List[Dict[str, Any]]) -> str:
    if not ranked:
        return ("No sample events fit that profile yet. Try selecting more goals or widening the filters.\n\n"
                f"**Search real events:** {real_search_links(profile)}")
    if not globals().get("LMSTUDIO_READY", False):
        return fallback_advice(profile, ranked) + "\n\n_LM Studio not connected - showing built-in advice._"
    system = ("You are a friendly, practical NYC career coach. Be concise and specific. Use only the events "
              "listed; never invent events, dates, prices or employers. You may name well-known real resources "
              "(school career centers, professional associations, LinkedIn, Handshake) without dates.")
    user = f"""
User profile:
- Job title or major: {profile['title_or_major'] or 'not given'}
- Year in college: {profile['college_year']}
- Looking for: {', '.join(profile['goal_labels']) or 'not specified'}
- Best time: {profile['best_time']}
- Extra notes: {profile['notes'] or 'none'}

Ranked sample events (best first):
{describe_matches(ranked)}

Write in Markdown, under 220 words:
1) One or two sentences on what this person should focus on right now given their year/role and goals.
2) "Top picks": the best 3 events from the list, each with one line on why and one concrete prep tip.
3) "Your intro": a one-sentence elevator pitch they could use at these events.
4) "Keep looking": 2-3 kinds of real organizations or professional associations in their field worth following.
"""
    try:
        text = lmstudio_chat(system, user)
    except RuntimeError as error:
        return fallback_advice(profile, ranked) + f"\n\n_LM Studio error, showing built-in advice: {error}_"
    return text + f"\n\n**Find real upcoming events:** {real_search_links(profile)}"

print("Advice functions ready. LM Studio connected:", globals().get("LMSTUDIO_READY", False))

## Cell 6 — Open the app
If the browser doesn't open automatically, click the local URL printed below.

In [ ]:
# === Cell 6: Gradio app (layout modeled on the reference design) ===
import gradio as gr

if not globals().get("MATCHER_READY", False):
    raise RuntimeError("Run Cells 3-5 successfully before opening the app.")
print("Gradio version:", gr.__version__)

CHIP_CHOICES = [("✦ All events", "All events"), ("💼 Job fairs", "Job fair"), ("🎤 Conferences", "Conference"),
                ("🤝 Networking", "Networking"), ("🛠️ Workshops", "Workshop"), ("🏫 Info sessions", "Info session")]
AGENT_HEADER = """
<div class="ce-agent-head"><div class="ce-agent-icon">✦</div>
  <div><b>Your career agent</b><small>Preference-driven demo</small></div></div>
<p class="ce-agent-intro">Tell your agent about your career and what you want. It ranks the events and explains why.</p>"""
START_TIP = ('<div class="ce-tip">Try <b>"Junior, computer science major"</b> + Internships, '
             'or <b>"Registered nurse"</b> + Job hunting on weekends.</div>')

def refresh_grid(search, borough, price, category, ranked, saved):
    return render_grid(filter_events(search, borough, price, category), ranked, saved or [])

def find_matches(goals, title_or_major, college_year, best_time, notes,
                 search, borough, price, category, saved):
    if not (title_or_major or "").strip():
        raise gr.Error("Add your job title or major first (e.g. 'Marketing coordinator' or 'Biology major').")
    if not goals:
        raise gr.Error("Pick at least one thing you're looking for (networking, job hunting, ...).")
    profile = build_profile(goals, title_or_major, college_year, best_time, notes)
    ranked = rank_events(profile, top_k=8)
    grid = render_grid(filter_events(search, borough, price, category), ranked, saved or [])
    hidden = len([m for m in ranked if EVENTS_BY_ID[m["id"]] not in filter_events(search, borough, price, category)])
    status = "⏳ Writing your personalized plan..."
    if hidden:
        status += f"\n\n_{hidden} of your matches are hidden by the search/borough/price/category filters._"
    return profile, ranked, grid, status

def write_advice(profile, ranked):
    if not profile:
        return START_TIP
    return build_career_advice(profile, ranked or [])

def show_all(search, borough, price, category, saved):
    return None, render_grid(filter_events(search, borough, price, category), None, saved or []), START_TIP

def save_from_card(token, saved, search, borough, price, category, ranked):
    try:
        event_id = int(str(token).split(":", 1)[0])
    except ValueError:
        return gr.update(), gr.update(), gr.update()
    saved = toggle_saved(saved, event_id)
    return saved, refresh_grid(search, borough, price, category, ranked, saved), render_saved(saved)

def save_from_dropdown(event_id, saved, search, borough, price, category, ranked):
    if event_id is None:
        return gr.update(), gr.update(), gr.update(), gr.update()
    saved = toggle_saved(saved, int(event_id))
    return saved, refresh_grid(search, borough, price, category, ranked, saved), render_saved(saved), None

try:
    gr.close_all()  # close an older copy if this cell is rerun
except Exception:
    pass

theme = gr.themes.Soft(primary_hue="indigo", neutral_hue="slate", radius_size="lg")
with gr.Blocks(title="NYC Career Event Finder", theme=theme, css=APP_CSS) as demo:
    saved_state = gr.State([])
    ranked_state = gr.State(None)
    profile_state = gr.State(None)

    gr.HTML(render_hero())
    with gr.Row(equal_height=False):
        # ---------------- Left: browse & filter ----------------
        with gr.Column(scale=7, elem_classes=["ce-panel"]):
            with gr.Row(elem_classes=["ce-filters"]):
                search_in = gr.Textbox(placeholder="Search a role, major, company, or event...",
                                       show_label=False, container=False, scale=5)
                borough_in = gr.Dropdown(["All boroughs"] + BOROUGHS, value="All boroughs",
                                         show_label=False, container=False, scale=3)
                price_in = gr.Dropdown(["Any price", "Free", "Paid"], value="Any price",
                                       show_label=False, container=False, scale=3)
            chips_in = gr.Radio(CHIP_CHOICES, value="All events", show_label=False,
                                container=False, elem_id="ce-chips")
            grid_out = gr.HTML(render_grid(EVENTS, None, []))

        # ---------------- Right: career agent ----------------
        with gr.Column(scale=3, min_width=320, elem_classes=["ce-panel"]):
            gr.HTML(AGENT_HEADER)
            goals_in = gr.CheckboxGroup(list(GOAL_OPTIONS), value=["Networking"], label="What are you looking for?")
            title_in = gr.Textbox(label="Job title or major",
                                  placeholder="e.g. Marketing coordinator, Nursing major")
            year_in = gr.Dropdown(COLLEGE_YEARS, value="Not in college", label="Year in college (if applicable)")
            time_in = gr.Dropdown(TIME_OPTIONS, value="Anytime", label="Best time")
            notes_in = gr.Textbox(label="Anything else? (optional)", lines=2,
                                  placeholder="e.g. I'd love fintech events after work")
            find_btn = gr.Button("✦ Find my matches", elem_id="ce-find-btn")
            reset_btn = gr.Button("Show all events", variant="secondary", size="sm")
            advice_out = gr.Markdown(START_TIP, elem_id="ce-advice")
            saved_out = gr.HTML(render_saved([]))
            save_pick = gr.Dropdown([(e["title"], e["id"]) for e in EVENTS], value=None,
                                    label="Save or remove an event by name")
            # Hidden bridge: the ☆ Save buttons on the cards write the event id here.
            save_box = gr.Textbox(elem_id="ce-save-box", show_label=False, container=False)

    filters = [search_in, borough_in, price_in, chips_in]
    for control in filters:
        control.change(refresh_grid, inputs=[*filters, ranked_state, saved_state], outputs=grid_out,
                       api_name=False, show_progress="hidden")

    find_btn.click(
        find_matches,
        inputs=[goals_in, title_in, year_in, time_in, notes_in, *filters, saved_state],
        outputs=[profile_state, ranked_state, grid_out, advice_out], api_name=False,
    ).success(write_advice, inputs=[profile_state, ranked_state], outputs=advice_out, api_name=False)

    reset_btn.click(show_all, inputs=[*filters, saved_state], outputs=[ranked_state, grid_out, advice_out],
                    api_name=False)
    save_box.input(save_from_card, inputs=[save_box, saved_state, *filters, ranked_state],
                   outputs=[saved_state, grid_out, saved_out], api_name=False, show_progress="hidden")
    save_pick.change(save_from_dropdown, inputs=[save_pick, saved_state, *filters, ranked_state],
                     outputs=[saved_state, grid_out, saved_out, save_pick], api_name=False,
                     show_progress="hidden")

demo.launch(share=False, inline=False, inbrowser=True, prevent_thread_lock=True,
            show_error=True, show_api=False)

## Cell 7 — Optional notebook-only test
Disabled during Run All.

In [ ]:
# === Cell 7: Optional test without the browser ===
# Leave False during Run All. Set True to try a profile right here in the notebook.
RUN_LOCAL_TEST = False

if RUN_LOCAL_TEST:
    from IPython.display import display, Markdown
    test_profile = build_profile(
        goal_labels=['Internships', 'Networking'],
        title_or_major='Junior, finance major',
        college_year='Junior',
        best_time='Weekday evenings',
        notes='interested in fintech',
    )
    test_ranked = rank_events(test_profile, top_k=5)
    display(pd.DataFrame([
        {'match': f"{m['percent']}%", **{k: EVENTS_BY_ID[m['id']][k] for k in ('title', 'category', 'borough', 'day', 'time')},
         'why': '; '.join(m['reasons'])}
        for m in test_ranked
    ]))
    display(Markdown(build_career_advice(test_profile, test_ranked)))
else:
    print('Optional test skipped. Use the app from Cell 6.')

## If a check stops

| Where it stops | What to do |
| --- | --- |
| Cell 1 says the Python version is wrong | Switch to **Python (Nutritionist 3.11)**. |
| Cell 1 says a restart is needed | Restart the kernel, then Run All. |
| Cell 2 prints a WARNING | LM Studio isn't reachable or has several models. The app still runs with built-in advice; fix LM Studio and rerun Cells 2 and 6 for AI-written plans. |
| Cell 4 assertion fails | You edited `EVENTS` or the keywords in Cell 3; check the new entries use the same keys and category names. |
| Cards' ☆ Save button does nothing | Use the **"Save or remove an event by name"** dropdown under Saved events — it does the same thing. |
| "Your top matches" shows fewer cards than expected | The search, borough, price or category filters are hiding some matches; clear them or click **Show all events**. |

## Customizing

* **Add real events:** append dictionaries to `EVENTS` in Cell 3 (same keys). `fields` uses the keys of
  `FIELD_LABELS` (or `"any"`); `goals` uses the values of `GOAL_OPTIONS`; `audience` is `students`,
  `early`, `experienced` or `all`; `slot` is `evening`, `weekend` or `daytime`.
* **Recognize more majors/titles:** add keywords to `FIELD_KEYWORDS` in Cell 3.
* **Change the look:** colors and spacing live in `APP_CSS` in Cell 4.